<!-- AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07 -->
# Task 2: fine-tuning a small code model to write docstrings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sathudeva7/CDAZZDEV-MLE-SATHURSAN/blob/main/task2_genai/task2_docstring_finetune.ipynb)

**Use case.** Qwen2.5-Coder-3B-Instruct is fine-tuned with QLoRA to write a Google-style
docstring for a Python function. The answer comes from the code itself, so
retrieval has nothing to add. The hard part is behaviour: one fixed layout, every
parameter and no others, `Yields` for generators, and only the exceptions the
code really raises.

This notebook is a thin driver. The logic lives in `task2_genai/` and carries offline
tests in `tests/test_task2_data.py`. Each section matches one rubric row, and every
number is printed by a code cell, so no figure in the text can drift from the run.

| Rubric row | Section | Where the logic lives |
|---|---|---|
| 2A Use case quality | 2A.1 | this notebook, `task2_genai/docstrings.py` |
| 2A Dataset size, teacher prompt | 2A.2, 2A.3 | `task2_genai/generate_data.py`, `prompts.py` |
| 2A Dataset diversity | 2A.4 | `task2_genai/recipes.py`, `metrics.py` |
| 2A Format and split | 2A.3, 2A.5 | `task2_genai/dataset.py` |
| 2B QLoRA implementation | 2B.2 | `task2_genai/finetune.py` |
| 2B Hyperparameter justification | 2B.1 | `finetune.HYPERPARAMETERS` |
| 2B Loss monitoring | 2B.3, 2B.4 | `finetune.build_trainer`, `epoch_losses` |
| 2B Model saved | 2B.5 | `finetune.merge_adapter` (`merge_and_unload()`) |
| 2C ROUGE-L comparison | 2C.1, 2C.3 | `metrics.rouge_l` |
| 2C Additional metric | 2C.4, 2C.5 | BERTScore; `metrics.structure_labels` |
| 2C Hallucination rate | 2C.6 | manual labels in this notebook |
| 2C Qualitative analysis | 2C.7 | this notebook |

**Models.** Teacher: OpenAI's GPT-6.1 Sol wrote the functions and their docstrings
(`task2_genai/generate_data.py`, run once locally; the data is committed). It is a paid
API, a deliberate exception to the free-tier setup (`docs/adr/0002-paid-teacher-for-task2.md`),
and a different model family from the student. Student: `Qwen/Qwen2.5-Coder-3B-Instruct`,
trained on Colab's free T4 GPU.

**To run it yourself:** Runtime → Change runtime type → T4 GPU. Add a Hugging Face
token with write access as `HF_TOKEN` under Colab's Secrets (the key icon) to publish
the merged model; without it the model is saved to the runtime's disk only. Then
Runtime → Run all, about an hour.

## Setup

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
import subprocess
import sys
import time
from pathlib import Path

RUN_STARTED = time.perf_counter()
REPO_URL = "https://github.com/sathudeva7/CDAZZDEV-MLE-SATHURSAN"
REPO_REF = "main"  # the branch or tag the Colab run clones
HF_REPO_NAME = "qwen2.5-coder-3b-docstrings"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content") / REPO_URL.rsplit("/", 1)[-1]
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "--quiet"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(ROOT / "task2_genai" / "requirements.txt")], check=True)
    RUN_DIR = Path("/content/t2_run")  # checkpoints and the 6 GB merged model stay out of the repo
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "task2_genai" else Path.cwd()
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
    RUN_DIR = ROOT / ".t2_run"
sys.path.insert(0, str(ROOT))

import bitsandbytes
import peft
import torch
import transformers

from common.llm import enable_console_logging
from common.llm_config import secret
from task2_genai.finetune import BASE_MODEL, SEED

enable_console_logging()
transformers.set_seed(SEED)
EVAL_DIR = ROOT / "task2_genai" / "evaluation"
EVAL_DIR.mkdir(exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()
print(f"Running in {'Colab' if IN_COLAB else 'a local kernel'}, repo at {commit or 'unknown commit'}")
print(f"torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}, bitsandbytes {bitsandbytes.__version__}")
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 1e9:.1f} GB")
print(f"Student: {BASE_MODEL}")
print(f"  {'✓' if secret('HF_TOKEN') else '✗'} HF_TOKEN")  # presence only, never the value

## 2A.1 Problem statement

**Input.** One Python function (5 to 40 lines) with no docstring: a plain function, a
method (with `self`), a generator or an `async def`.

**Output.** Only the docstring text, in this fixed Google-style layout:

```
Apply a percentage discount to a price, optionally capped.

Args:
    price: Original price.
    pct: Discount percentage, from 0 to 100.
    cap: Largest discount allowed; no cap when None.

Returns:
    The price after the discount.

Raises:
    ValueError: If pct is outside 0-100.
```

**What counts as correct.** `task2_genai/docstrings.py` reads the function with Python's
`ast` module and checks the docstring against it:

| Label | Meaning |
|---|---|
| correct | Every real parameter documented (never `self`/`cls`) and no others; `Returns` or `Yields` matches the code; `Raises` lists exactly the exceptions in the function's own `raise` statements; a true one-line summary. |
| partially correct | Nothing invented, but something missing: a parameter, a section, or a vague summary. |
| hallucinated | Mentions a parameter, exception, return value or yield the code does not have. Invented beats missing. |

The `ast` check covers parameters and sections automatically. Whether the summary is
true is judged by hand in 2C.6.

**Why fine-tuning and not RAG.** Everything the answer needs is in the input, so
there is nothing to retrieve. What a small model gets wrong is behaviour: it mixes
styles, documents `self`, writes `Returns` for generators and lists exceptions it
assumes might happen. That is what fine-tuning changes. A tuned 3B model also runs
locally, so private code never leaves the machine, at a fraction of a large model's cost.

## 2A.2 Teacher model and its prompts

Each example takes two teacher calls (`task2_genai/generate_data.py`):

1. **Functions.** One call per recipe cell (topic × kind) writes two new functions at
   each of three levels. Fresh code means the student cannot have memorised the
   answers during pretraining.
2. **Docstring.** A second call sees **only the code**, exactly as the student will, and
   fills the docstring as fields that our code renders into one fixed layout. The
   label therefore describes the code as written, not what the teacher meant it to do.

The full prompts follow (the brief asks for them), then the teacher's call log.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
import pandas as pd

from task2_genai.prompts import STUDENT_SYSTEM, TEACHER_DOCSTRING, TEACHER_FUNCTIONS

for title, text in [
    ("Teacher call 1, system", TEACHER_FUNCTIONS.system),
    ("Teacher call 1, user template", TEACHER_FUNCTIONS.user),
    ("Teacher call 2, system", TEACHER_DOCSTRING.system),
    ("Teacher call 2, user template", TEACHER_DOCSTRING.user),
    ("Student system prompt (base and fine-tuned alike)", STUDENT_SYSTEM),
]:
    print(f"===== {title} =====\n{text}\n")

calls = pd.read_json(ROOT / "task2_genai" / "logs" / "teacher_calls.jsonl", lines=True)
print("Teacher call log (task2_genai/logs/teacher_calls.jsonl):")
print(calls.groupby(["prompt", "model", "outcome"]).agg(
    calls=("ts", "size"), prompt_tokens=("prompt_tokens", "sum"), completion_tokens=("completion_tokens", "sum")
).to_string())

## 2A.3 Dataset size, filtering and split

Every function the teacher wrote is validated (exactly one `def`, the requested kind,
5 to 40 lines, no docstring, a name not seen before). Every docstring is labelled by
the `ast` check, and only `correct` ones are kept, so a teacher mistake never becomes
a training target. The kept examples are split 80/10/10, stratified by kind
(`dataset.split`, seed 42).

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
import re
from collections import Counter

from task2_genai.dataset import DATA_DIR, SPLITS, read_jsonl

functions = read_jsonl(DATA_DIR / "functions.jsonl")
rejected = read_jsonl(DATA_DIR / "rejected.jsonl")
labels = read_jsonl(DATA_DIR / "labels.jsonl")
splits = {name: read_jsonl(DATA_DIR / f"{name}.jsonl") for name in SPLITS}
kept = sum(len(rows) for rows in splits.values())
label_counts = Counter(row["label"] for row in labels)

print(pd.DataFrame({
    "step": ["Functions written by the teacher (call 1)", "Passed validation",
             "Docstrings written by the teacher (call 2)", "Docstrings passing the ast check (kept)"],
    "examples": [len(functions) + len(rejected), len(functions), len(labels), label_counts["correct"]],
}).to_string(index=False))

reasons = Counter(re.sub(r" '.*'|: .*|\d+", "", row["reason"]).strip() for row in rejected)
print("\nRejected functions by reason:", dict(reasons))
print("Docstring labels from the ast check:", dict(label_counts))
for row in [r for r in labels if r["label"] != "correct"][:3]:
    print(f"  dropped {row['id']}: {row['label']}, missing {row['missing']}, invented {row['invented']}")

sizes = pd.DataFrame({"split": SPLITS, "examples": [len(splits[s]) for s in SPLITS]})
sizes["share"] = (sizes.examples / kept).map("{:.0%}".format)
print(f"\n{kept} examples kept, split as:")
print(sizes.to_string(index=False))
assert kept == label_counts["correct"]

## 2A.4 Diversity

Variety is built in by the recipe grid (`task2_genai/recipes.py`): 12 topics × 4 kinds ×
3 levels. This section measures it: prompt lengths in the student's own tokens, the grid
coverage, keyword frequencies, and the most similar pair of functions in the whole set.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
import matplotlib.pyplot as plt

from task2_genai.finetune import load_tokenizer

tokenizer = load_tokenizer()
records = [dict(r, split=name) for name in SPLITS for r in splits[name]]


def n_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


stats = pd.DataFrame([{
    "id": r["id"], "topic": r["topic"], "kind": r["kind"], "level": r["level"], "split": r["split"],
    "code_tokens": n_tokens(r["messages"][1]["content"]),
    "docstring_tokens": n_tokens(r["messages"][2]["content"]),
    "total_tokens": n_tokens(tokenizer.apply_chat_template(r["messages"], tokenize=False)),
} for r in records])
print(stats[["code_tokens", "docstring_tokens", "total_tokens"]].describe().round(0).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for level in ["simple", "medium", "tricky"]:
    subset = stats[stats.level == level]
    axes[0].hist(subset.code_tokens, bins=20, alpha=0.6, label=level)
    axes[1].hist(subset.docstring_tokens, bins=20, alpha=0.6, label=level)
axes[0].set(title="Prompt length (the function), tokens", xlabel="tokens", ylabel="examples")
axes[1].set(title="Answer length (the docstring), tokens", xlabel="tokens")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
from task2_genai import metrics

print("Examples per topic and kind:")
print(pd.crosstab(stats.topic, stats.kind, margins=True).to_string())
print("\nExamples per level:", stats.level.value_counts().to_dict())

codes = [r["messages"][1]["content"] for r in records]
ids = [r["id"] for r in records]
print("\nTop 20 words in function names:", metrics.name_words(codes).most_common(20))
print("Standard-library modules called (functions using each):", metrics.modules_used(codes).most_common(15))

pairs = metrics.near_duplicates(ids, codes)
print(f"\nMost similar pairs (Jaccard similarity of character 5-grams; {metrics.NEAR_DUPLICATE_JACCARD} would mean near copies):")
for a, b, sim in pairs:
    print(f"  {sim:.2f}  {a}  vs  {b}")
print("No near-duplicates in the dataset" if pairs[0][2] < metrics.NEAR_DUPLICATE_JACCARD else "Near-duplicates found")

## 2A.5 JSONL format and chat template

Each line of `task2_genai/data/{train,val,test}.jsonl` holds the recipe fields and a
`messages` list of system, user and assistant turns. At training time the tokenizer's
own chat template renders it into Qwen's format. One training example, exactly as the
model sees it:

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
example = splits["train"][0]
print({k: v for k, v in example.items() if k != "messages"})
print(tokenizer.apply_chat_template(example["messages"], tokenize=False))

## 2B.1 Hyperparameters

Every setting is in `task2_genai/finetune.py` with the reason it was chosen; the table
below is printed from that code. The two lengths are then checked against the data.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
from task2_genai.finetune import HYPERPARAMETERS, hp

table = pd.DataFrame([{"parameter": name, "value": str(h.value), "reason": h.reason} for name, h in HYPERPARAMETERS.items()])
with pd.option_context("display.max_colwidth", None):
    display(table)

longest = int(stats.total_tokens.max())
longest_answer = int(stats.docstring_tokens.max())
print(f"Longest example: {longest} tokens, max_seq_length {hp('max_seq_length')}: nothing is truncated")
print(f"Longest reference docstring: {longest_answer} tokens, max_new_tokens {hp('max_new_tokens')}")
assert longest <= hp("max_seq_length") and longest_answer < hp("max_new_tokens")

## 2C.1 Baseline answers, before any training

The base model answers the test set first, with the same system prompt the fine-tuned
model gets and greedy decoding. Generating these now means only one model is on the
T4 at a time.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
from task2_genai.finetune import free_gpu, generate, load_fp16

test = splits["test"]
started = time.perf_counter()
base_model = load_fp16(BASE_MODEL)
base_answers = generate(base_model, tokenizer, test)
del base_model
free_gpu()
print(f"{len(base_answers)} baseline answers in {time.perf_counter() - started:.0f} s. First one ({test[0]['id']}):\n")
print(base_answers[0])

## 2B.2 QLoRA model

The student is loaded in 4-bit NF4 with double quantization, prepared for k-bit training
(`prepare_model_for_kbit_training`), and given LoRA adapters on every linear layer
(`finetune.load_quantized_model`). Only the adapters train.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
from task2_genai.finetune import load_quantized_model

model = load_quantized_model()
quant = model.config.quantization_config
print(f"4-bit: {quant.load_in_4bit}, type: {quant.bnb_4bit_quant_type}, double quant: {quant.bnb_4bit_use_double_quant}, "
      f"compute dtype: {quant.bnb_4bit_compute_dtype}")
for name, module in model.named_modules():
    if name.endswith("layers.0.self_attn.q_proj"):
        print(f"{name}: {type(module).__name__} wrapping {type(module.base_layer).__name__}")
model.print_trainable_parameters()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2B.3 Training

Each example is tokenized with the prompt masked out of the labels, so the loss covers
only the docstring and its end token (`finetune.tokenize_record`). The Trainer evaluates
on the validation split after every epoch and keeps the best epoch.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
from task2_genai.finetune import IGNORE_INDEX, build_trainer, tokenize_record

tokenizer.padding_side = "right"
train_rows = [tokenize_record(r, tokenizer) for r in splits["train"]]
val_rows = [tokenize_record(r, tokenizer) for r in splits["val"]]
answer_tokens = sum(sum(label != IGNORE_INDEX for label in row["labels"]) for row in train_rows)
all_tokens = sum(len(row["input_ids"]) for row in train_rows)
print(f"{len(train_rows)} train and {len(val_rows)} validation examples; "
      f"the loss covers {answer_tokens} of {all_tokens} training tokens ({answer_tokens / all_tokens:.0%})")

trainer = build_trainer(model, tokenizer, train_rows, val_rows, RUN_DIR / "checkpoints")
started = time.perf_counter()
trainer.train()
print(f"Trained in {(time.perf_counter() - started) / 60:.1f} min; peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 2B.4 Training and validation loss per epoch

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
import json

from task2_genai.finetune import epoch_losses

losses = pd.DataFrame(epoch_losses(trainer.state.log_history))
print(losses.round(4).to_string(index=False))
val_decreasing = bool((losses.val_loss.diff().dropna() < 0).all())
print(f"\nValidation loss decreases every epoch: {val_decreasing}")
print(f"Best validation loss {trainer.state.best_metric:.4f}, kept from {trainer.state.best_model_checkpoint}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(losses.epoch, losses.train_loss, marker="o", label="train (mean over the epoch)")
ax.plot(losses.epoch, losses.val_loss, marker="o", label="validation (end of epoch)")
ax.set(xlabel="epoch", ylabel="loss (docstring tokens)", xticks=losses.epoch, title="QLoRA training loss")
ax.legend()
plt.tight_layout()
fig.savefig(EVAL_DIR / "loss_per_epoch.png", dpi=120)
plt.show()
(EVAL_DIR / "loss_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))

## 2B.5 Merge and publish

The best adapter is saved, then folded into the base model with `merge_and_unload()`.
The merge uses the float16 base, not the 4-bit training copy, so no quantization error
is baked into the saved weights. The merged model is pushed to the Hugging Face Hub
with a model card and the base model's licence.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
import shutil

from task2_genai.finetune import merge_adapter

ADAPTER_DIR = RUN_DIR / "adapter"
MERGED_DIR = RUN_DIR / "merged"
trainer.model.save_pretrained(ADAPTER_DIR)  # the best epoch, reloaded by load_best_model_at_end
del model, trainer
free_gpu()

merged = merge_adapter(ADAPTER_DIR)
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
size_gb = sum(f.stat().st_size for f in MERGED_DIR.glob("*")) / 1e9
print(f"Merged model saved to {MERGED_DIR} ({size_gb:.1f} GB)")

MODEL_URL = None
hf_token = secret("HF_TOKEN")
if hf_token:
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi(token=hf_token)
    repo_id = f"{api.whoami()['name']}/{HF_REPO_NAME}"
    shutil.copy(hf_hub_download(BASE_MODEL, "LICENSE"), MERGED_DIR / "LICENSE")
    (MERGED_DIR / "README.md").write_text(
        "---\nbase_model: " + BASE_MODEL + "\nlicense: other\nlicense_name: qwen-research\n"
        "license_link: LICENSE\nlibrary_name: transformers\ntags: [qlora, docstrings, code]\n---\n\n"
        "# Qwen2.5-Coder-3B-Instruct, fine-tuned to write Google-style docstrings\n\n"
        f"QLoRA fine-tune of `{BASE_MODEL}` (adapters merged with `merge_and_unload()`), trained on "
        f"{len(splits['train'])} teacher-written Python functions with ast-checked docstrings. "
        "Send one function as the user message with the system prompt from "
        "[task2_genai/prompts.py](" + REPO_URL + "/blob/main/task2_genai/prompts.py); "
        "the reply is the docstring text. Training and evaluation: " + REPO_URL + "/tree/main/task2_genai\n"
    )
    api.create_repo(repo_id, exist_ok=True, private=False)
    api.upload_folder(folder_path=str(MERGED_DIR), repo_id=repo_id, commit_message="Merged QLoRA fine-tune for docstrings")
    MODEL_URL = f"https://huggingface.co/{repo_id}"
    print("Published:", MODEL_URL)
else:
    print("HF_TOKEN is not set, so the merged model is only on this runtime's disk:", MERGED_DIR)

## 2B.6 Out-of-memory notes

_Filled in after the submission run._

## 2C.2 Fine-tuned answers

The merged model answers the same test set with the same system prompt and decoding
as the baseline in 2C.1.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
started = time.perf_counter()
ft_answers = generate(merged, tokenizer, test)
del merged
free_gpu()
print(f"{len(ft_answers)} fine-tuned answers in {time.perf_counter() - started:.0f} s. First one ({test[0]['id']}):\n")
print(ft_answers[0])

## 2C.3 ROUGE-L on the held-out test set

ROUGE-L F1 against the teacher's checked docstring (Google's `rouge-score`, Porter
stemming), both models on the identical test set. Code fences and wrapping quotes are
stripped from both models' answers alike (`docstrings.clean_output`).

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
references = [r["messages"][2]["content"] for r in test]
scores = pd.DataFrame({
    "id": [r["id"] for r in test], "kind": [r["kind"] for r in test], "level": [r["level"] for r in test],
    "base_rougeL": metrics.rouge_l(base_answers, references),
    "ft_rougeL": metrics.rouge_l(ft_answers, references),
})
print(f"Test set: {len(test)} functions, identical for both models\n")
print(pd.DataFrame({
    "model": ["Base (system prompt, no fine-tuning)", "Fine-tuned (QLoRA, merged)"],
    "ROUGE-L F1 mean": [scores.base_rougeL.mean(), scores.ft_rougeL.mean()],
    "median": [scores.base_rougeL.median(), scores.ft_rougeL.median()],
    "min": [scores.base_rougeL.min(), scores.ft_rougeL.min()],
}).round(3).to_string(index=False))
print("\nMean ROUGE-L F1 by kind:")
print(scores.groupby("kind")[["base_rougeL", "ft_rougeL"]].mean().round(3).to_string())

## 2C.4 BERTScore F1

BERTScore compares meaning rather than exact words (`roberta-large`, the package's default
for English), so a correct docstring worded differently from the reference is not
punished as hard as under ROUGE-L. Scores are rescaled with the package's baseline, which
spreads them over a readable range.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
from bert_score import score as bert_score

from task2_genai.docstrings import clean_output

for column, answers in [("base_bertscore", base_answers), ("ft_bertscore", ft_answers)]:
    _, _, f1 = bert_score([clean_output(a) for a in answers], references, lang="en", rescale_with_baseline=True, verbose=False)
    scores[column] = f1.tolist()
print(pd.DataFrame({
    "model": ["Base (system prompt, no fine-tuning)", "Fine-tuned (QLoRA, merged)"],
    "BERTScore F1 mean": [scores.base_bertscore.mean(), scores.ft_bertscore.mean()],
    "median": [scores.base_bertscore.median(), scores.ft_bertscore.median()],
}).round(3).to_string(index=False))

## 2C.5 Structure check on every test answer

The same `ast` check that filtered the training data labels every answer from both
models. It cannot judge whether the summary is true (2C.6 does that by hand), but it
catches every invented parameter, exception or section without anyone reading the answer.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
codes_test = [r["messages"][1]["content"] for r in test]
base_checks = metrics.structure_labels(codes_test, base_answers)
ft_checks = metrics.structure_labels(codes_test, ft_answers)
scores["base_check"] = [c["label"] for c in base_checks]
scores["ft_check"] = [c["label"] for c in ft_checks]

order = ["correct", "partial", "hallucinated", "unparseable"]
check_table = pd.DataFrame({
    "Base": scores.base_check.value_counts().reindex(order, fill_value=0),
    "Fine-tuned": scores.ft_check.value_counts().reindex(order, fill_value=0),
})
print(check_table.to_string())
for name, column in [("Base", "base_check"), ("Fine-tuned", "ft_check")]:
    share = (scores[column] == "hallucinated").mean()
    print(f"{name}: {share:.0%} of answers invent a parameter, exception or section")
print("\nWhat the base model got wrong most often:",
      Counter(item.split(" ")[0] + " " + kind for c in base_checks for kind in ("missing", "invented") for item in c[kind]).most_common(5))

## 2C.6 Manual review and hallucination rate

Twelve test answers from the fine-tuned model, picked at random with the notebook's seed
before reading any answers. Each is labelled by hand as **correct**, **partial** or
**hallucinated** using the definitions in 2A.1, including whether the summary is true,
which the `ast` check cannot see. The checker's label is shown as a hint only.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
import random

REVIEW_COUNT = 12
review_ids = [test[i]["id"] for i in sorted(random.Random(SEED).sample(range(len(test)), REVIEW_COUNT))]
by_id = {r["id"]: i for i, r in enumerate(test)}
for rid in review_ids:
    i = by_id[rid]
    check = ft_checks[i]
    print(f"{'=' * 30} {rid} {'=' * 30}\n{codes_test[i]}\n{'-' * 20} fine-tuned answer {'-' * 20}\n{ft_answers[i]}")
    print(f"{'-' * 20} checker: {check['label']}, missing {check['missing']}, invented {check['invented']}\n")

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
# Filled in by hand after reading the answers above: (label, note), label one of correct / partial / hallucinated.
MANUAL_LABELS: dict[str, tuple[str, str]] = {
}

unlabelled = [rid for rid in review_ids if rid not in MANUAL_LABELS]
if unlabelled:
    print("Label these first:", unlabelled)
else:
    review = pd.DataFrame([{"id": rid, "label": MANUAL_LABELS[rid][0], "note": MANUAL_LABELS[rid][1],
                            "checker": ft_checks[by_id[rid]]["label"]} for rid in review_ids])
    with pd.option_context("display.max_colwidth", None):
        display(review)
    counts = review.label.value_counts()
    hallucinated = int(counts.get("hallucinated", 0))
    print(dict(counts))
    print(f"Hallucination rate: {hallucinated / len(review):.0%} ({hallucinated} of {len(review)} reviewed answers)")

## 2C.7 Qualitative analysis

_Written after the submission run, from the answers above._

## Run outputs

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 2 notebook: QLoRA training, merge, and evaluation against the base model', Date: 2026-10-07
summary = {
    "student": BASE_MODEL,
    "model_url": MODEL_URL,
    "split_sizes": {s: len(splits[s]) for s in SPLITS},
    "loss_per_epoch": losses.to_dict(orient="records"),
    "rougeL_f1_mean": {"base": scores.base_rougeL.mean(), "fine_tuned": scores.ft_rougeL.mean()},
    "bertscore_f1_mean": {"base": scores.base_bertscore.mean(), "fine_tuned": scores.ft_bertscore.mean()},
    "structure_check": check_table.to_dict(),
    "manual_review": MANUAL_LABELS,
    "runtime_min": round((time.perf_counter() - RUN_STARTED) / 60, 1),
}
(EVAL_DIR / "results.json").write_text(json.dumps(summary, indent=1))
with (EVAL_DIR / "predictions.jsonl").open("w") as fh:
    for i, r in enumerate(test):
        fh.write(json.dumps({"id": r["id"], "code": codes_test[i], "reference": references[i],
                             "base": base_answers[i], "fine_tuned": ft_answers[i],
                             **{k: scores.iloc[i][k] for k in ["base_rougeL", "ft_rougeL", "base_bertscore", "ft_bertscore", "base_check", "ft_check"]}}) + "\n")
print(json.dumps({k: v for k, v in summary.items() if k != "manual_review"}, indent=1, default=str))

if IN_COLAB:
    shutil.make_archive("/content/task2_run_outputs", "zip", EVAL_DIR)
    from google.colab import files
    files.download("/content/task2_run_outputs.zip")